# Movie Sentiment Analysis (NLP)
Classifying IMDB movie reviews as Positive or Negative using NLP techniques.

**Dataset:** IMDB 50K Movie Reviews (Kaggle)

**Models Used:** Logistic Regression, Naive Bayes

**Best Result:** Logistic Regression — Accuracy: 88%+

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully!')

## 2. Load Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('IMDB Dataset.csv')
print('Shape:', df.shape)
print('\nSentiment distribution:')
print(df['sentiment'].value_counts())
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Sentiment distribution chart
plt.figure(figsize=(6, 4))
counts = df['sentiment'].value_counts()
plt.bar(counts.index, counts.values, color=['#22c55e', '#ef4444'], edgecolor='white', width=0.5)
plt.title('Sentiment Distribution', fontsize=13, fontweight='bold')
plt.xlabel('Sentiment')
plt.ylabel('Count')
for i, v in enumerate(counts.values):
    plt.text(i, v + 100, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Review length analysis
df['review_length'] = df['review'].apply(len)

plt.figure(figsize=(8, 4))
for sentiment, color in [('positive', '#22c55e'), ('negative', '#ef4444')]:
    subset = df[df['sentiment'] == sentiment]['review_length']
    plt.hist(subset, bins=50, alpha=0.6, color=color, label=sentiment)
plt.title('Review Length Distribution by Sentiment', fontsize=13, fontweight='bold')
plt.xlabel('Review Length (characters)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('review_length.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Lowercase
    text = text.lower()
    # Remove stopwords
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    return ' '.join(tokens)

print('Cleaning reviews... (this may take a minute)')
df['cleaned_review'] = df['review'].apply(clean_text)
print('Done!')
print('\nOriginal:', df['review'][0][:200])
print('\nCleaned: ', df['cleaned_review'][0][:200])

## 5. TF-IDF Vectorisation

In [ ]:
# Encode labels
df['label'] = (df['sentiment'] == 'positive').astype(int)

X = df['cleaned_review']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TF-IDF vectorisation
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')
print(f'Vocabulary size  : {len(tfidf.vocabulary_)}')

## 6. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes':         MultinomialNB()
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    results[name] = round(acc * 100, 2)
    print(f'{name:22s} | Accuracy: {acc*100:.2f}%')

## 7. Confusion Matrix

In [ ]:
# Best model confusion matrix
best_model = models['Logistic Regression']
preds = best_model.predict(X_test_tfidf)
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix — Logistic Regression', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_test, preds, target_names=['Negative', 'Positive']))

## 8. Test on Your Own Review

In [ ]:
def predict_sentiment(review_text):
    cleaned = clean_text(review_text)
    vectorized = tfidf.transform([cleaned])
    prediction = best_model.predict(vectorized)[0]
    confidence = best_model.predict_proba(vectorized)[0].max()
    label = 'POSITIVE' if prediction == 1 else 'NEGATIVE'
    print(f'Sentiment  : {label}')
    print(f'Confidence : {confidence*100:.1f}%')

# Try it out!
predict_sentiment("This movie was absolutely fantastic! The acting was brilliant and the story kept me hooked throughout.")
print()
predict_sentiment("Terrible film. Waste of time, boring plot and bad acting. Would not recommend to anyone.")